# nl2sh+ QLoRA training on Colab or Kaggle

Select a GPU runtime before running. This notebook trains only the pinned NL2SH-ALFA train split, saves an adapter and a metadata record, and never uploads your files or asks for a Hub token. Training can take hours; download the final ZIP before the free runtime resets.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU found. In Colab choose Runtime > Change runtime type > GPU; in Kaggle choose a GPU accelerator.')
gpu = torch.cuda.get_device_properties(0)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(gpu.total_memory / 1024**3, 1))
print('PyTorch:', torch.__version__)

In [ ]:
import pathlib, subprocess, sys
repo_url = 'https://github.com/HariHaran9597/nl2sh-Junior-Shell-Assistant.git'
candidates = [pathlib.Path('/content/nl2sh'), pathlib.Path('/kaggle/working/nl2sh')]
repo = next((path for path in candidates if (path / 'training' / 'train_qlora.py').exists()), candidates[0] if pathlib.Path('/content').exists() else candidates[1])
if not (repo / 'training' / 'train_qlora.py').exists():
    repo.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--depth', '1', repo_url, str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'training' / 'requirements.txt')], check=True)
print('Training source:', repo / 'training' / 'train_qlora.py')

## Full run

The default is one epoch, effective batch size 8 on one GPU, sequence length 512, QLoRA rank 16, and seed 42. Training resumes from the newest checkpoint if the runtime filesystem still has it.

In [ ]:
import pathlib, subprocess, sys
work = pathlib.Path('/content/nl2sh-training') if pathlib.Path('/content').exists() else pathlib.Path('/kaggle/working/nl2sh-training')
script = repo / 'training' / 'train_qlora.py'
subprocess.run([sys.executable, str(script), '--output-dir', str(work)], check=True)
print('Adapter folder:', work / 'final_adapter')

In [ ]:
import pathlib, zipfile
adapter = work / 'final_adapter'
if not (adapter / 'adapter_config.json').exists():
    raise FileNotFoundError(f'Adapter missing: {adapter}. Do not report this run as complete.')
archive = work.parent / 'nl2sh-qlora-adapter.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(adapter.rglob('*')):
        if path.is_file():
            zf.write(path, path.relative_to(work.parent))
print('Download this artifact from the runtime file browser:', archive)
print('It includes training_metadata.json; save it with the adapter.')

## Optional: export a GGUF for the local CLI

This merges the adapter, builds a pinned CPU quantizer, and creates Q4_K_M. It needs several additional gigabytes of free runtime disk and can take a while. Run it after training; it does not replace the currently published model.

In [ ]:
export_dir = work.parent / 'nl2sh-gguf-export'
export_script = repo / 'training' / 'export_gguf.py'
subprocess.run([sys.executable, str(export_script), '--adapter', str(work / 'final_adapter'), '--output-dir', str(export_dir)], check=True)
print('GGUF:', export_dir / 'nl2sh-q4_k_m.gguf')
print('Metadata:', export_dir / 'gguf_metadata.json')